In [2]:
# Neural Identifier Training with Particle Filters - Simple Pendulum

import numpy as np
import plotly.graph_objects as go

# ============================================================
# 1) True nonlinear system (Simple Pendulum)
# ============================================================
def plant_dynamics(x, u):
    """
    Continuous dynamics for a Simple Pendulum with Damping.
    x = [theta, theta_dot] (Angle, Angular Velocity)
    
    Equation:
    theta_doubledot = -(g/L) * sin(theta) - (b / (m*L^2)) * theta_dot + u
    
    Parameters assumed:
    g = 9.81 m/s^2
    L = 1.0 m
    m = 1.0 kg
    b = 0.5 (damping coefficient)
    """
    g = 9.81
    L = 1.0
    m = 1.0
    b = 0.5
    
    theta, theta_dot = x
    
    # Pendulum equations
    x1_dot = theta_dot
    # Dynamics: Gravity term + Damping term + Input torque (scaled)
    x2_dot = -(g / L) * np.sin(theta) - (b / (m * L**2)) * theta_dot + (1.0 / (m * L**2)) * u
    
    return np.array([x1_dot, x2_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-3):
    """
    One Euler step of the discrete plant with process noise.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est):
    """
    Features for the Pendulum.
    Even though the physics contains sin(theta), the RHONN approximates it 
    using high-order sigmoid terms (Taylor series approximation concept).
    
    z = [S(x1), S(x2), S(x1)S(x2), S(x1)^2, S(x2)^2, S(x1)^3, x1, x2, 1]
    """
    s_x1 = sigmoidal(x_est[0])  # x1 (angle)
    s_x2 = sigmoidal(x_est[1])  # x2 (angular velocity)
    
    return np.array([
        s_x1, s_x2,                           # Sigmoid terms
        s_x1*s_x2,                            # Cross term
        s_x1**2, s_x2**2,                     # Quadratic sigmoid terms
        s_x1**3,                              # Cubic term (helps approx sin(x))
        x_est[0], x_est[1],                   # Linear terms
        1.0                                   # Bias
    ])


def RHONN_predict(x_state_for_z, w_neuron):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) , u(k) )
    """
    z_i = construct_z_vector(x_state_for_z)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # Series-Parallel: use measured angle

        z_i = construct_z_vector(x_state_for_z)          
        H_i = z_i.reshape(-1, 1)                          

        for i in range(self.num_neurons):
            P_pred = self.P[i] + self.Q[i]
            P_pred += np.eye(self.num_weights_per_neuron) * 1e-8

            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10: M_i = 1e-10

            x_hat_pred_i = self.weights[i] @ z_i
            e_i = chi_kp1[i] - x_hat_pred_i
            e_i = np.clip(e_i, -10.0, 10.0)

            K_i = (P_pred @ H_i).flatten() / M_i

            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.1))
            self.weights[i] += adaptive_eta * K_i * e_i

            I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-6

# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=0.05, R_std=0.1, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]
        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)
        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1
        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous):
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0] 
        z = construct_z_vector(x_state_for_z)

        # 1) Predict
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std

        # 2) Update
        for i in range(self.num_neurons):
            w_mat = self.particles[i]
            x_pred_particles = w_mat @ z
            innov = chi_kp1[i] - x_pred_particles
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll)
            like = np.exp(ll)
            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]


# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights
# ============================================================
class UKF_RHONN_Trainer:
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        
        self.alpha = alpha
        self.beta = beta
        self.kappa = kappa if kappa is not None else 3 - num_weights_per_neuron
        
        self.n = num_weights_per_neuron
        self.lambda_ = alpha**2 * (self.n + self.kappa) - self.n
        
        self.Wm = np.zeros(2 * self.n + 1)
        self.Wc = np.zeros(2 * self.n + 1)
        self.Wm[0] = self.lambda_ / (self.n + self.lambda_)
        self.Wc[0] = self.lambda_ / (self.n + self.lambda_) + (1 - alpha**2 + beta)
        for i in range(1, 2 * self.n + 1):
            self.Wm[i] = 1.0 / (2 * (self.n + self.lambda_))
            self.Wc[i] = 1.0 / (2 * (self.n + self.lambda_))

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)
            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance):
        n = len(mean)
        sigma_points = np.zeros((2 * n + 1, n))
        sigma_points[0] = mean
        try:
            sqrt = np.linalg.cholesky((n + self.lambda_) * covariance)
        except np.linalg.LinAlgError:
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + self.lambda_)
        
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        return sigma_points

    def _measurement_function(self, weight_sigma_point, z_vector):
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous):
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0] 

        z_i = construct_z_vector(x_state_for_z)

        for i in range(self.num_neurons):
            sigma_points = self._generate_sigma_points(self.weights[i], self.P[i])
            predicted_sigma_points = sigma_points.copy()
            
            predicted_mean = np.sum(self.Wm[:, np.newaxis] * predicted_sigma_points, axis=0)
            predicted_cov = self.Q[i].copy()
            for j in range(2 * self.n + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += self.Wc[j] * np.outer(diff, diff)
            
            measurement_sigma_points = np.zeros(2 * self.n + 1)
            for j in range(2 * self.n + 1):
                measurement_sigma_points[j] = self._measurement_function(predicted_sigma_points[j], z_i)
            
            predicted_measurement = np.sum(self.Wm * measurement_sigma_points)
            innovation_cov = self.R[i][0]
            for j in range(2 * self.n + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += self.Wc[j] * diff_meas**2
            
            cross_cov = np.zeros(self.n)
            for j in range(2 * self.n + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += self.Wc[j] * diff_state * diff_meas
            
            if innovation_cov < 1e-12: innovation_cov = 1e-12
            K = cross_cov / innovation_cov
            
            innovation = chi_kp1[i] - predicted_measurement
            self.weights[i] = predicted_mean + self.eta * K * innovation
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov
            
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.n) * 1e-6

# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 1000
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'laplacian' 
    process_noise_std = 0.05 # Reduced slightly for Pendulum

    # --- True system init ---
    x_true = np.zeros((n_steps, 2))
    # Initial conditions: [Angle (rad), Velocity (rad/s)]
    # Start at 90 degrees (pi/2) approx 1.57 rad
    x_true[0] = [1.5, 0.0] 
    
    # --- Input u ---
    # We can apply a small torque or leave it 0 (damped free oscillation)
    # Let's apply a small sinusoidal torque to make identification harder
    u_input = 0.5 * np.sin(3 * t_history) 

    # --- RHONN config ---
    num_neurons = 2  # Two states
    num_features = 9 # Feature vector size
    num_weights_per_neuron = num_features

    # --- Common initial weights ---
    common_initial_weights = [np.random.uniform(-0.2, 0.2, num_weights_per_neuron) for _ in range(num_neurons)]
    print("Common Initial Weights initialized.")

    # --- EKF ---
    ekf_trainer = EKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=1e-3, R_init=1e-2, P_init=1.0, eta=0.6
    )
    x_hat_ekf = np.zeros((n_steps, 2))
    x_hat_ekf[0] = x_true[0]

    # --- UKF ---
    ukf_trainer = UKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=1e-3, R_init=1e-2, P_init=1.0, eta=0.8,
        alpha=1e-3, beta=2.0 
    )
    x_hat_ukf = np.zeros((n_steps, 2))
    x_hat_ukf[0] = x_true[0]

    # --- PF ---
    n_particles = 100
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        n_particles=n_particles,
        initial_weights=common_initial_weights,
        Q_std=0.2, R_std=0.5, ess_threshold=n_particles / 2 
    )

    # Init particles around common weights
    for i in range(pf_trainer.num_neurons):
        pf_trainer.particles[i] = np.tile(
            common_initial_weights[i], (pf_trainer.n_particles, 1)
        )
        # Add slight jitter
        pf_trainer.particles[i] += np.random.randn(pf_trainer.n_particles, num_weights_per_neuron) * 0.05
        pf_trainer.weights_pf[i] = np.ones(pf_trainer.n_particles) / pf_trainer.n_particles

    x_hat_pf = np.zeros((n_steps, 2))
    x_hat_pf[0] = x_true[0]

    print("Starting Pendulum simulation...")
    for k in range(n_steps - 1):
        u_k = u_input[k]
        
        # ---- 1) true system -> k+1 ----
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std)

        # ---- 2) EKF update ----
        ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ekf[k])
        x_state_for_z_ekf = np.copy(x_hat_ekf[k])
        x_state_for_z_ekf[0] = x_true[k][0] # SP
        x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0])
        x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1])

        # ---- 2b) UKF update ----
        ukf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ukf[k])
        x_state_for_z_ukf = np.copy(x_hat_ukf[k])
        x_state_for_z_ukf[0] = x_true[k][0] # SP
        x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[0])
        x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[1])

        # ---- 3) PF update ----
        pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_pf[k])
        pf_weight_estimates = pf_trainer.get_estimate()
        x_state_for_z_pf = np.copy(x_hat_pf[k])
        x_state_for_z_pf[0] = x_true[k][0] # SP
        x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0])
        x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1])

        if k % (n_steps // 10) == 0:
            print(f"Simulation progress: {k/n_steps*100:.1f}%")

    print("Simulación finalizada.")

    # ============================================================
    # 6) Resultados y Gráficas - Formato Tesis
    # ============================================================
    
    # Configuración de formato para tesis
    thesis_config = {
        'font_family': 'Computer Modern, serif',
        'font_size': 14,
        'title_font_size': 16,
        'legend_font_size': 12,
        'line_width_true': 2.5,
        'line_width_est': 2.0,
        'plot_width': 1000,
        'plot_height': 500,
        'grid_color': 'rgba(200, 200, 200, 0.3)',
        'grid_width': 0.5
    }
    
    # Cálculo de MSE por estado
    mse_theta_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
    mse_omega_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
    mse_theta_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)
    mse_omega_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)
    mse_theta_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)
    mse_omega_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)
    
    mse_total_ekf = mse_theta_ekf + mse_omega_ekf
    mse_total_ukf = mse_theta_ukf + mse_omega_ukf
    mse_total_pf = mse_theta_pf + mse_omega_pf
    
    # Reporte MSE
    print("\n" + "="*70)
    print("🏆 MEJOR FILTRO: ", end="")
    mse_dict = {'EKF': mse_total_ekf, 'UKF': mse_total_ukf, 'PF': mse_total_pf}
    best_filter = min(mse_dict, key=mse_dict.get)
    print(f"{best_filter} (MSE total: {mse_dict[best_filter]:.6f})")
    print("="*70)
    
    print("\n--- Comparación de Desempeño (MSE) - Péndulo Simple ---")
    print(f"EKF MSE θ (ángulo):       {mse_theta_ekf:.6f}")
    print(f"EKF MSE ω (velocidad):    {mse_omega_ekf:.6f}")
    print(f"UKF MSE θ (ángulo):       {mse_theta_ukf:.6f}")
    print(f"UKF MSE ω (velocidad):    {mse_omega_ukf:.6f}")
    print(f"PF  MSE θ (ángulo):       {mse_theta_pf:.6f}")
    print(f"PF  MSE ω (velocidad):    {mse_omega_pf:.6f}")
    
    # Gráficas individuales por estado
    states_info = [
        {'idx': 0, 'var': 'θ', 'desc': 'Ángulo', 'y_label': 'Ángulo θ (rad)'},
        {'idx': 1, 'var': 'ω', 'desc': 'Velocidad Angular', 'y_label': 'Velocidad Angular ω (rad/s)'}
    ]
    
    for state_info in states_info:
        i = state_info['idx']
        
        fig = go.Figure()
        
        # Estado real (línea negra gruesa)
        fig.add_trace(go.Scatter(
            x=t_history, y=x_true[:, i],
            mode='lines',
            name='Estado Real',
            line=dict(color='#000000', width=thesis_config['line_width_true']),
            showlegend=True
        ))
        
        # Estimación EKF
        fig.add_trace(go.Scatter(
            x=t_history, y=x_hat_ekf[:, i],
            mode='lines',
            name='EKF-RHONN',
            line=dict(color='#1f77b4', width=thesis_config['line_width_est'], dash='dash'),
            showlegend=True
        ))
        
        # Estimación UKF
        fig.add_trace(go.Scatter(
            x=t_history, y=x_hat_ukf[:, i],
            mode='lines',
            name='UKF-RHONN',
            line=dict(color='#2ca02c', width=thesis_config['line_width_est'], dash='dot'),
            showlegend=True
        ))
        
        # Estimación PF
        fig.add_trace(go.Scatter(
            x=t_history, y=x_hat_pf[:, i],
            mode='lines',
            name='PF-RHONN',
            line=dict(color='#d62728', width=thesis_config['line_width_est'], dash='dashdot'),
            showlegend=True
        ))
        
        fig.update_layout(
            title={
                'text': f'Estado {state_info["var"]}: {state_info["desc"]} - Péndulo Simple',
                'x': 0.5,
                'xanchor': 'center',
                'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
            },
            xaxis_title='Tiempo (s)',
            yaxis_title=state_info['y_label'],
            xaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            yaxis=dict(
                showgrid=True,
                gridcolor=thesis_config['grid_color'],
                gridwidth=thesis_config['grid_width'],
                showline=True,
                linewidth=1.5,
                linecolor='black',
                mirror=True,
                ticks='outside',
                tickwidth=1.5,
                ticklen=5
            ),
            legend=dict(
                x=0.02,
                y=0.98,
                xanchor='left',
                yanchor='top',
                bgcolor='rgba(255, 255, 255, 0.9)',
                bordercolor='black',
                borderwidth=1,
                font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
            ),
            font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
            plot_bgcolor='white',
            paper_bgcolor='white',
            width=thesis_config['plot_width'],
            height=thesis_config['plot_height'],
            margin=dict(l=80, r=40, t=80, b=60)
        )
        
        fig.show()
    
    # Gráfica de errores combinada
    error_theta_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
    error_omega_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
    error_theta_ukf = x_true[:, 0] - x_hat_ukf[:, 0]
    error_omega_ukf = x_true[:, 1] - x_hat_ukf[:, 1]
    error_theta_pf = x_true[:, 0] - x_hat_pf[:, 0]
    error_omega_pf = x_true[:, 1] - x_hat_pf[:, 1]
    
    fig_err = go.Figure()
    
    # Errores θ (ángulo)
    fig_err.add_trace(go.Scatter(
        x=t_history, y=error_theta_ekf,
        mode='lines',
        name=f'EKF Error θ (MSE={mse_theta_ekf:.2e})',
        line=dict(color='#1f77b4', width=1.5),
        opacity=0.8
    ))
    
    fig_err.add_trace(go.Scatter(
        x=t_history, y=error_theta_ukf,
        mode='lines',
        name=f'UKF Error θ (MSE={mse_theta_ukf:.2e})',
        line=dict(color='#2ca02c', width=1.5),
        opacity=0.8
    ))
    
    fig_err.add_trace(go.Scatter(
        x=t_history, y=error_theta_pf,
        mode='lines',
        name=f'PF Error θ (MSE={mse_theta_pf:.2e})',
        line=dict(color='#d62728', width=1.5),
        opacity=0.8
    ))
    
    # Errores ω (velocidad)
    fig_err.add_trace(go.Scatter(
        x=t_history, y=error_omega_ekf,
        mode='lines',
        name=f'EKF Error ω (MSE={mse_omega_ekf:.2e})',
        line=dict(color='#1f77b4', width=1.5, dash='dot'),
        opacity=0.8
    ))
    
    fig_err.add_trace(go.Scatter(
        x=t_history, y=error_omega_ukf,
        mode='lines',
        name=f'UKF Error ω (MSE={mse_omega_ukf:.2e})',
        line=dict(color='#2ca02c', width=1.5, dash='dot'),
        opacity=0.8
    ))
    
    fig_err.add_trace(go.Scatter(
        x=t_history, y=error_omega_pf,
        mode='lines',
        name=f'PF Error ω (MSE={mse_omega_pf:.2e})',
        line=dict(color='#d62728', width=1.5, dash='dot'),
        opacity=0.8
    ))
    
    fig_err.update_layout(
        title={
            'text': 'Errores de Estimación - Péndulo Simple',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tiempo (s)',
        yaxis_title='Error de Estimación',
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig_err.show()
    
    # Espacio de fase (Retrato de Fase del Péndulo)
    fig_phase = go.Figure()
    
    fig_phase.add_trace(go.Scatter(
        x=x_true[:, 0], y=x_true[:, 1],
        mode='lines',
        name='Retrato de Fase Real',
        line=dict(color='#000000', width=3)
    ))
    
    fig_phase.add_trace(go.Scatter(
        x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1],
        mode='lines',
        name='Estimación EKF-RHONN',
        line=dict(color='#1f77b4', width=2, dash='dash')
    ))
    
    fig_phase.add_trace(go.Scatter(
        x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1],
        mode='lines',
        name='Estimación UKF-RHONN',
        line=dict(color='#2ca02c', width=2, dash='dot')
    ))
    
    fig_phase.add_trace(go.Scatter(
        x=x_hat_pf[:, 0], y=x_hat_pf[:, 1],
        mode='lines',
        name='Estimación PF-RHONN',
        line=dict(color='#d62728', width=2, dash='dashdot')
    ))
    
    fig_phase.update_layout(
        title={
            'text': 'Espacio de Fases - Péndulo Simple (θ vs ω)',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Ángulo θ (rad)',
        yaxis_title='Velocidad Angular ω (rad/s)',
        xaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_width'],  # Aspecto cuadrado
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig_phase.show()
    
    # Gráfica de barras comparando MSE
    fig_mse = go.Figure()
    
    filters = ['EKF-RHONN', 'UKF-RHONN', 'PF-RHONN']
    
    fig_mse.add_trace(go.Bar(
        name='Ángulo θ',
        x=filters,
        y=[mse_theta_ekf, mse_theta_ukf, mse_theta_pf],
        marker_color='#636EFA',
        text=[f'{mse_theta_ekf:.2e}', f'{mse_theta_ukf:.2e}', f'{mse_theta_pf:.2e}'],
        textposition='outside'
    ))
    
    fig_mse.add_trace(go.Bar(
        name='Velocidad Angular ω',
        x=filters,
        y=[mse_omega_ekf, mse_omega_ukf, mse_omega_pf],
        marker_color='#EF553B',
        text=[f'{mse_omega_ekf:.2e}', f'{mse_omega_ukf:.2e}', f'{mse_omega_pf:.2e}'],
        textposition='outside'
    ))
    
    fig_mse.update_layout(
        title={
            'text': 'Comparación de Error Cuadrático Medio (MSE)',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': thesis_config['title_font_size'], 'family': thesis_config['font_family']}
        },
        xaxis_title='Tipo de Filtro',
        yaxis_title='Error Cuadrático Medio (MSE)',
        yaxis=dict(
            type='log',
            showgrid=True,
            gridcolor=thesis_config['grid_color'],
            gridwidth=thesis_config['grid_width'],
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        xaxis=dict(
            showline=True,
            linewidth=1.5,
            linecolor='black',
            mirror=True,
            ticks='outside',
            tickwidth=1.5,
            ticklen=5
        ),
        legend=dict(
            x=0.02,
            y=0.98,
            xanchor='left',
            yanchor='top',
            bgcolor='rgba(255, 255, 255, 0.9)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=thesis_config['legend_font_size'], family=thesis_config['font_family'])
        ),
        barmode='group',
        font=dict(size=thesis_config['font_size'], family=thesis_config['font_family']),
        plot_bgcolor='white',
        paper_bgcolor='white',
        width=thesis_config['plot_width'],
        height=thesis_config['plot_height'],
        margin=dict(l=80, r=40, t=80, b=60)
    )
    
    fig_mse.show()

Common Initial Weights initialized.
Starting Pendulum simulation...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulation progress: 40.0%
Simulation progress: 50.0%
Simulation progress: 60.0%
Simulation progress: 70.0%
Simulation progress: 80.0%
Simulation progress: 90.0%
Simulación finalizada.

🏆 MEJOR FILTRO: UKF (MSE total: 0.002082)

--- Comparación de Desempeño (MSE) - Péndulo Simple ---
EKF MSE θ (ángulo):       0.003495
EKF MSE ω (velocidad):    0.002184
UKF MSE θ (ángulo):       0.001072
UKF MSE ω (velocidad):    0.001010
PF  MSE θ (ángulo):       0.015158
PF  MSE ω (velocidad):    0.012665
